In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers accelerate
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.41.2 tokenizers==0.19.1 datasets accelerate tqdm scikit-learn

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM
from datasets import load_dataset, concatenate_datasets, load_from_disk

BASE_DIR = "/kaggle/working/polyglot_ai_detector_v2"
DATASET_PATH = os.path.join(BASE_DIR, "temporal_polyglot_ds") 
SHARD_DIR = os.path.join(BASE_DIR, "temporal_shards") 
MAX_LEN = 256 

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GPUS = torch.cuda.device_count()

os.makedirs(SHARD_DIR, exist_ok=True)
os.makedirs(DATASET_PATH, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base-mlm")
model = AutoModelForMaskedLM.from_pretrained("microsoft/codebert-base-mlm")

if NUM_GPUS > 1:
    model = nn.DataParallel(model)

model = model.to(DEVICE)
model.eval()

processed_shards = []
SHARD_SIZE = total_samples // NUM_SHARDS

for i in range(NUM_SHARDS):
    shard_path = os.path.join(SHARD_DIR, f"shard_{i}")
    
    if os.path.exists(shard_path):
        processed_shards.append(load_from_disk(shard_path))
        continue

    start_idx = i * SHARD_SIZE
    end_idx = min((i + 1) * SHARD_SIZE, total_samples)
    current_shard = ds.select(range(start_idx, end_idx))
    
    processed_shard = current_shard.map(
        calculate_metrics_batched, 
        batched=True, 
        batch_size=64,
        desc=f"Shard {i}"
    )
    
    processed_shard.save_to_disk(shard_path)
    processed_shards.append(processed_shard)

final_ds = concatenate_datasets(processed_shards)
final_ds.save_to_disk(DATASET_PATH)

print(f"Precomputation complete. Total samples: {len(final_ds)}")

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset, random_split
from transformers import RobertaTokenizer, T5EncoderModel
from datasets import load_from_disk
from tqdm.auto import tqdm
import torch.multiprocessing

torch.multiprocessing.set_sharing_strategy('file_system')

MODEL_NAME = "Salesforce/codet5-base"
BATCH_SIZE = 48
ACCUM_STEPS = 2
EPOCHS = 2
MAX_LR = 5e-5
MAX_LEN = 256
METRIC_DIM = 7

BASE_DIR = "/kaggle/working/polyglot_ai_detector_v2"
SHARD_DIR = os.path.join(BASE_DIR, "temporal_shards")
SAVE_PATH = os.path.join(BASE_DIR, "polyglot_temporal_final.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_EVERY = 1000
NUM_CORES = 4

shard_paths = [os.path.join(SHARD_DIR, f"shard_{i}") for i in range(50)]
processed_shards = [load_from_disk(p) for p in shard_paths if os.path.exists(p)]
ds = ConcatDataset(processed_shards)

test_size = int(0.05 * len(ds))
train_ds, val_ds = random_split(ds, [len(ds) - test_size, test_size], generator=torch.Generator().manual_seed(42))

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

def custom_collate(batch):
    tokenized = tokenizer(
        [item["code"] for item in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    raw_metrics = torch.tensor([item["metric_vector"] for item in batch], dtype=torch.float32)
    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": torch.tensor([item["label"] for item in batch], dtype=torch.long),
        "metric_vector": torch.nan_to_num(raw_metrics, nan=0.0, posinf=10.0, neginf=-100.0)
    }

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=custom_collate, num_workers=NUM_CORES, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=custom_collate, pin_memory=True)

base_model = T5EncoderModel.from_pretrained(MODEL_NAME)
for p in base_model.parameters(): p.requires_grad = False
for name, p in base_model.named_parameters():
    if any(f"block.{i}" in name for i in [8, 9, 10, 11]): p.requires_grad = True

class MageCodeClassifier(nn.Module):
    def __init__(self, base, metric_dim=METRIC_DIM):
        super().__init__()
        self.base = base
        h = base.config.hidden_size
        self.metric_cnn = nn.Sequential(
            nn.Conv1d(metric_dim, 32, 3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(h + 64, 1024), nn.ReLU(), nn.Dropout(0.1), nn.Linear(1024, 1)
        )

    def forward(self, input_ids, attention_mask, metric_vector):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-4)
        cnn_features = self.metric_cnn(metric_vector.transpose(1, 2)).squeeze(-1)
        return self.classifier(torch.cat([pooled, cnn_features], dim=1))

model = MageCodeClassifier(base_model)
if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=MAX_LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, total_steps=(len(train_loader) // ACCUM_STEPS) * EPOCHS, pct_start=0.1
)
loss_fn = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

def evaluate():
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= 100: break
            with torch.amp.autocast("cuda"):
                logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE), batch["metric_vector"].to(DEVICE)).view(-1)
            all_logits.append(logits.cpu()); all_labels.append(batch["labels"].cpu())
    
    preds = (torch.sigmoid(torch.cat(all_logits)) > 0.5).int()
    labels = torch.cat(all_labels).int()
    tp, fp, fn = ((preds==1)&(labels==1)).sum(), ((preds==1)&(labels==0)).sum(), ((preds==0)&(labels==1)).sum()
    model.train()
    return 2 * tp / (2 * tp + fp + fn + 1e-8)

global_step, best_f1 = 0, 0.0
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
        with torch.amp.autocast("cuda"):
            logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE), batch["metric_vector"].to(DEVICE)).view(-1)
            loss = loss_fn(logits, batch["labels"].float().to(DEVICE)) / ACCUM_STEPS
        
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        global_step += 1
        if global_step % CHECKPOINT_EVERY == 0:
            val_f1 = evaluate()
            if val_f1 > best_f1: best_f1 = val_f1
            print(f"Step {global_step} | Val F1: {val_f1:.4f} | Best: {best_f1:.4f}")

torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(), SAVE_PATH)